In [ ]:
# import math
# from dataclasses import dataclass

# # ---------- Physical constants ----------
# hbar = 1.054_571_817e-34      # J·s
# e    = 1.602_176_634e-19      # C
# pi   = math.pi

# @dataclass
# class ResonatorParams:
#     f_r_Hz: float                 # Resonator frequency (Hz)
#     C_mode_F: float | None = None # Mode capacitance (F), if known. Otherwise provide Cprime and length.
#     Cprime_F_per_m: float | None = None  # CPW per-unit-length capacitance (F/m), optional
#     length_m: float | None = None        # Physical length of resonator (m), optional
#     geometry: str = "lambda4"     # "lambda4" or "lambda2" (used only if C_mode_F is None)

# @dataclass
# class TransmonParams:
#     C_sigma_F: float              # Total qubit capacitance CΣ (F)
#     C_c_F: float                  # Qubit–resonator coupling capacitance (F)
#     EJ_over_EC: float             # Ratio EJ/EC (dimensionless)

# @dataclass
# class OptionalDispersive:
#     # Only needed if you want chi
#     f_q_Hz: float | None = None   # Qubit transition frequency (Hz)
#     anharmonicity_Hz: float | None = None  # Transmon anharmonicity alpha (~ -E_C/h) in Hz

# def mode_capacitance(res: ResonatorParams) -> float:
#     """
#     Returns the resonator mode capacitance C_mode (F).
#     If C_mode is supplied directly, uses it.
#     Otherwise, computes from CPW per-unit capacitance and length:
#         lambda/4: C_mode ≈ (C' * l)/2
#         lambda/2: C_mode ≈ (C' * l)
#     """
#     if res.C_mode_F is not None:
#         return res.C_mode_F
#     if res.Cprime_F_per_m is None or res.length_m is None:
#         raise ValueError("Either provide C_mode_F, or both Cprime_F_per_m and length_m.")
#     if res.geometry.lower() in ("lambda4", "λ/4", "quarter"):
#         return 0.5 * res.Cprime_F_per_m * res.length_m
#     elif res.geometry.lower() in ("lambda2", "λ/2", "half"):
#         return res.Cprime_F_per_m * res.length_m
#     else:
#         raise ValueError("geometry must be 'lambda4' or 'lambda2'")

# def vrms(f_r_Hz: float, C_mode_F: float) -> float:
#     """Zero-point voltage of the resonator mode."""
#     omega_r = 2 * pi * f_r_Hz
#     return math.sqrt(hbar * omega_r / (2.0 * C_mode_F))

# def n_ge_analytic(EJ_over_EC: float) -> float:
#     """
#     Transmon-limit analytic estimate of |<g|n|e>|:
#         n_zpf ≈ (EJ / (32 * EC))^(1/4)
#     """
#     return (EJ_over_EC / 32.0) ** 0.25

# def coupling_g(transmon: TransmonParams, res: ResonatorParams) -> tuple[float, float]:
#     """
#     Returns (g_rad_s, g_Hz).
#     g = (2e/ħ) * (C_c/CΣ) * V_rms * |<g|n|e>|
#     """
#     C_mode = mode_capacitance(res)
#     V_rms  = vrms(res.f_r_Hz, C_mode)
#     beta   = transmon.C_c_F / transmon.C_sigma_F
#     n_ge   = n_ge_analytic(transmon.EJ_over_EC)
#     g_rad  = (2.0 * e / hbar) * beta * V_rms * n_ge
#     return g_rad, g_rad / (2.0 * pi)

# def dispersive_shift_chi(g_rad_s: float, f_q_Hz: float, f_r_Hz: float, anharmonicity_Hz: float) -> tuple[float, float]:
#     """
#     Transmon dispersive shift (second-order):
#         chi = - g^2 * alpha / [ Δ (Δ + alpha) ]
#     where Δ = ω_q - ω_r and alpha = 2π * anharmonicity_Hz.
#     Returns (chi_rad_s, chi_Hz).
#     """
#     omega_q = 2 * pi * f_q_Hz
#     omega_r = 2 * pi * f_r_Hz
#     alpha   = 2 * pi * anharmonicity_Hz
#     Delta   = omega_q - omega_r
#     chi_rad = - (g_rad_s**2) * alpha / (Delta * (Delta + alpha))
#     return chi_rad, chi_rad / (2.0 * pi)

# # ---------------- Example usage ----------------
# if __name__ == "__main__":
#     # --- Fill in with YOUR numbers ---
#     res = ResonatorParams(
#         f_r_Hz=6.75e9,                 # 7 GHz resonator
#         Cprime_F_per_m=163e-12,       # e.g., 160 pF/m (depends on CPW geometry & substrate)
#         length_m=3.29e-3,              # 5.5 mm physical length
#         geometry="lambda4"
#     )

#     transmon = TransmonParams(
#         C_sigma_F=96.4e-15,             # 80 fF total qubit capacitance
#         C_c_F=4.55e-15,                # 2 fF coupling cap between qubit island and resonator
#         EJ_over_EC=82.16               # canonical transmon ratio
#     )

#     g_rad, g_Hz = coupling_g(transmon, res)
#     print(f"g / 2π = {g_Hz/1e6:.2f} MHz   (g = {g_rad:.3e} rad/s)")

#     # (Optional) If you want chi, provide qubit f_q and anharmonicity (usually alpha ≈ -E_C/h)
#     # Suppose f_q = 5.5 GHz and alpha = -250 MHz:
#     f_q = 4.9e9
#     alpha = -162e6
#     chi_rad, chi_Hz = dispersive_shift_chi(g_rad, f_q, res.f_r_Hz, alpha)
#     print(f"chi / 2π = {chi_Hz/1e6:.2f} MHz   (chi = {chi_rad:.3e} rad/s)")


g / 2π = 83.44 MHz   (g = 5.243e+08 rad/s)
chi / 2π = 0.30 MHz   (chi = 1.904e+06 rad/s)


In [20]:
import math
from dataclasses import dataclass

# ---------- Physical constants ----------
hbar = 1.054_571_817e-34      # J·s
e    = 1.602_176_634e-19      # C
pi   = math.pi

@dataclass
class ResonatorParams:
    f_r_Hz: float
    C_mode_F: float | None = None
    Cprime_F_per_m: float | None = None
    length_m: float | None = None
    geometry: str = "lambda4"   # "lambda4" (single-ended) or "lambda2"

@dataclass
class TransmonParams:
    C_sigma_F: float
    C_c_F: float
    EJ_over_EC: float

# ---------------- Core helpers (unchanged) ----------------
def mode_capacitance(res: ResonatorParams) -> float:
    if res.C_mode_F is not None:
        return res.C_mode_F
    if res.Cprime_F_per_m is None or res.length_m is None:
        raise ValueError("Either provide C_mode_F, or both Cprime_F_per_m and length_m.")
    g = res.geometry.lower()
    if g in ("lambda4", "λ/4", "quarter"):
        return 0.5 * res.Cprime_F_per_m * res.length_m
    elif g in ("lambda2", "λ/2", "half"):
        return res.Cprime_F_per_m * res.length_m
    else:
        raise ValueError("geometry must be 'lambda4' or 'lambda2'")

def vrms(f_r_Hz: float, C_mode_F: float) -> float:
    omega_r = 2 * pi * f_r_Hz
    return math.sqrt(hbar * omega_r / (2.0 * C_mode_F))

def n_ge_analytic(EJ_over_EC: float) -> float:
    return (EJ_over_EC / 32.0) ** 0.25

def coupling_g(transmon: TransmonParams, res: ResonatorParams) -> tuple[float, float]:
    C_mode = mode_capacitance(res)
    V_rms  = vrms(res.f_r_Hz, C_mode)
    beta   = transmon.C_c_F / transmon.C_sigma_F
    n_ge   = n_ge_analytic(transmon.EJ_over_EC)
    g_rad  = (2.0 * e / hbar) * beta * V_rms * n_ge
    return g_rad, g_rad / (2.0 * pi)

def dispersive_shift_chi(g_rad_s: float, f_q_Hz: float, f_r_Hz: float, anharmonicity_Hz: float) -> tuple[float, float]:
    omega_q = 2 * pi * f_q_Hz
    omega_r = 2 * pi * f_r_Hz
    alpha   = 2 * pi * anharmonicity_Hz
    Delta   = omega_q - omega_r
    chi_rad = - (g_rad_s**2) * alpha / (Delta * (Delta + alpha))
    return chi_rad, chi_rad / (2.0 * pi)

# ---------------- New: Q-factors & linewidths ----------------
def external_Q(res: ResonatorParams, Cc_feed_F: float, R_L_ohm: float = 50.0) -> float:
    """
    External Q for a CPW resonator capacitively coupled in series to a load R_L.
    Uses the standard small-coupling result (Göppl et al., JAP 2008):
        lambda/4 (single-ended): Q_ext ≈ C_mode / (ω R_L Cc^2)
        lambda/2 (two-port, symmetric): Q_ext ≈ C_mode / (2 ω R_L Cc^2)
    """
    omega = 2 * pi * res.f_r_Hz
    C_mode = mode_capacitance(res)
    g = res.geometry.lower()
    if g in ("lambda4", "λ/4", "quarter"):
        return C_mode / (omega * R_L_ohm * Cc_feed_F**2)
    elif g in ("lambda2", "λ/2", "half"):
        return C_mode / (2.0 * omega * R_L_ohm * Cc_feed_F**2)
    else:
        raise ValueError("geometry must be 'lambda4' or 'lambda2'")

def kappa_from_Q(f_Hz: float, Q: float) -> tuple[float, float]:
    """Return (kappa_rad_s, kappa_Hz) from frequency and Q."""
    omega = 2 * pi * f_Hz
    kappa = omega / Q
    return kappa, kappa / (2.0 * pi)

def loaded_Q(Q_ext: float | None, Q_int: float | None) -> float | None:
    """
    1/Q_L = 1/Q_ext + 1/Q_int.
    Provide at least one; returns None if both are None.
    """
    if Q_ext is None and Q_int is None:
        return None
    inv = 0.0
    if Q_ext is not None and Q_ext > 0:
        inv += 1.0 / Q_ext
    if Q_int is not None and Q_int > 0:
        inv += 1.0 / Q_int
    return math.inf if inv == 0 else 1.0 / inv

def purcell_T1_from_res(g_rad_s: float, kappa_tot_rad_s: float, Delta_rad_s: float) -> float:
    """
    Simple Purcell-limited T1 estimate:
        1/T1 ≈ (g^2 / Δ^2) * κ_tot   (valid in dispersive regime)
    Returns T1 in seconds.
    """
    if Delta_rad_s == 0:
        return math.inf
    gamma = (g_rad_s**2 / (Delta_rad_s**2)) * kappa_tot_rad_s
    return math.inf if gamma == 0 else 1.0 / gamma

# ---------------- Example with YOUR numbers ----------------
if __name__ == "__main__":
    res = ResonatorParams(
        f_r_Hz=7.25e9,
        Cprime_F_per_m=165e-12,
        length_m=3.063e-3,
        geometry="lambda4",
    )

    transmon = TransmonParams(
        C_sigma_F=96.3e-15,
        C_c_F=4.55e-15,
        EJ_over_EC=82.16,
    )

    # --- Coupling g ---
    g_rad, g_Hz = coupling_g(transmon, res)
    print(f"g / 2π = {g_Hz/1e6:.2f} MHz   (g = {g_rad:.3e} rad/s)")

    # --- Dispersive shift (optional) ---
    f_q = 4.9e9
    alpha = -162e6
    chi_rad, chi_Hz = dispersive_shift_chi(g_rad, f_q, res.f_r_Hz, alpha)
    print(f"chi / 2π = {chi_Hz/1e6:.2f} MHz   (chi = {chi_rad:.3e} rad/s)")

    # --- Q-factors / linewidths ---
    # Provide your feedline coupling capacitor here (between resonator and 50Ω line).
    # This is NOT the qubit coupling capacitance.
    Cc_feed_F = 3.70e-15   # e.g. 3.0e-15  # <-- fill this

    # If you have an estimate/measurement for internal Q (dielectric + conductor + radiation)
    Q_int = 30e5       # e.g. 1.0e5     # <-- optional

    if Cc_feed_F is not None:
        Q_ext = external_Q(res, Cc_feed_F, R_L_ohm=50.0)
        kext_rad, kext_Hz = kappa_from_Q(res.f_r_Hz, Q_ext)
        print(f"Q_ext = {Q_ext:,.0f}   κ_ext/2π = {kext_Hz/1e6:.3f} MHz")

        # Loaded Q/linewidth if Q_int is given
        Q_L = loaded_Q(Q_ext, Q_int)
        if Q_L is not None and math.isfinite(Q_L):
            kL_rad, kL_Hz = kappa_from_Q(res.f_r_Hz, Q_L)
            print(f"Q_loaded = {Q_L:,.0f}   κ_loaded/2π = {kL_Hz/1e6:.3f} MHz")
        elif Q_L is math.inf:
            print("Q_loaded = ∞ (no losses specified)")

        # (Optional) Purcell T1 using total κ (needs Q_L or κ_tot)
        if Q_L is not None and math.isfinite(Q_L):
            k_tot_rad, _ = kappa_from_Q(res.f_r_Hz, Q_L)
        else:
            # Fall back to external only if that's all we know
            k_tot_rad = kext_rad
        Delta_rad = 2 * pi * (f_q - res.f_r_Hz)
        T1_P = purcell_T1_from_res(g_rad, k_tot_rad, Delta_rad)
        print(f"Purcell T1 (estimate) = {T1_P*1e6:.1f} µs")
    else:
        print("Set Cc_feed_F (feedline coupler) to compute Q_ext, κ, and Purcell T1.")


g / 2π = 89.17 MHz   (g = 5.603e+08 rad/s)
chi / 2π = 0.22 MHz   (chi = 1.371e+06 rad/s)
Q_ext = 8,104   κ_ext/2π = 0.895 MHz
Q_loaded = 8,082   κ_loaded/2π = 0.897 MHz
Purcell T1 (estimate) = 123.2 µs
